In [11]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from typing import List, Dict, Tuple
from collections import OrderedDict

In [12]:
# Weighted Federated Average

def federated_average(
    client_weights: List[Dict[str, np.ndarray]],
    client_sizes: List[int]
) -> Dict[str, np.ndarray]:
    """
    Aggregate client model weights using weighted FedAvg.

    Args:
        client_weights : list of state_dicts (numpy arrays) from each client
        client_sizes   : number of training samples per client

    Returns:
        Aggregated global model weights (same structure as input dicts)
    """
    total_samples = sum(client_sizes)
    scaling_factors = [n / total_samples for n in client_sizes]

    # Initialize accumulator from first client
    aggregated = {k: np.zeros_like(v) for k, v in client_weights[0].items()}

    for weight_dict, scale in zip(client_weights, scaling_factors):
        for key in aggregated:
            aggregated[key] += scale * weight_dict[key]

    return aggregated

In [13]:
# Server Class
class FederatedServer:
    def __init__(self, global_model_weights: Dict[str, np.ndarray]):
        """
        Args:
            global_model_weights: initial model weights (state dict as numpy arrays)
        """
        self.global_weights = copy.deepcopy(global_model_weights)
        self.round_history: List[Dict] = []

    # ── Aggregation ──────────────────────────
    def aggregate(
        self,
        client_updates: List[Tuple[Dict[str, np.ndarray], int]]
    ) -> Dict[str, np.ndarray]:
        """
        Run one round of FedAvg aggregation and update the global model.

        Args:
            client_updates: list of (client_weights, num_samples) tuples

        Returns:
            Updated global weights
        """
        weights_list = [w for w, _ in client_updates]
        sizes_list   = [s for _, s in client_updates]

        self.global_weights = federated_average(weights_list, sizes_list)

        self.round_history.append({
            "num_clients": len(client_updates),
            "total_samples": sum(sizes_list),
            "client_sample_sizes": sizes_list,
        })

        return self.global_weights

    # ── Update & Redistribution ───────────────
    def update_and_redistribute(
        self,
        client_updates: List[Tuple[Dict[str, np.ndarray], int]],
        client_ids: List[str],
    ) -> Dict[str, Dict[str, np.ndarray]]:
        """
        Aggregate client updates to form a new global model, then
        immediately redistribute the updated weights to all clients.

        Args:
            client_updates: list of (client_weights, num_samples) tuples
            client_ids    : list of client identifiers to redistribute to

        Returns:
            Mapping of client_id → updated global_weights copy
        """
        # Step 1 — Update global model via FedAvg
        self.aggregate(client_updates)
        print(f"  [Server] Global model updated "
              f"(round {len(self.round_history)}, "
              f"{len(client_updates)} clients, "
              f"{sum(s for _, s in client_updates):,} samples)")

        # Step 2 — Redistribute updated weights to every client
        redistributed = self.distribute(client_ids)
        print(f"  [Server] Updated weights redistributed → {client_ids}")

        return redistributed

    # ── Distribution ─────────────────────────
    def get_global_model(self) -> Dict[str, np.ndarray]:
        """Return a deep copy of the current global model for distribution."""
        return copy.deepcopy(self.global_weights)

    def distribute(
        self, client_ids: List[str]
    ) -> Dict[str, Dict[str, np.ndarray]]:
        """
        Distribute the global model to all specified clients.

        Args:
            client_ids: list of client identifiers

        Returns:
            Mapping of client_id → global_weights copy
        """
        return {cid: self.get_global_model() for cid in client_ids}

    # ── Utilities ────────────────────────────
    def summary(self) -> None:
        print(f"Rounds completed : {len(self.round_history)}")
        for i, r in enumerate(self.round_history, 1):
            print(f"  Round {i}: {r['num_clients']} clients, "
                  f"{r['total_samples']} total samples")


In [14]:
#Pytorch Adapter (Optional)
def pytorch_state_dict_to_numpy(state_dict) -> Dict[str, np.ndarray]:
    """Convert a PyTorch OrderedDict state_dict to numpy arrays."""
    return {k: v.cpu().numpy() for k, v in state_dict.items()}

def numpy_to_pytorch_state_dict(numpy_dict: Dict[str, np.ndarray]):
    """Convert numpy weight dict back to a PyTorch-compatible OrderedDict."""
    import torch
    return OrderedDict({k: torch.tensor(v) for k, v in numpy_dict.items()})

In [15]:
# ── Data Loading & Preprocessing ──────────────────────────────────────────────
df = pd.read_csv("diabetes_prediction_dataset.csv")

# Encode categorical columns
le_gender = LabelEncoder()
le_smoking = LabelEncoder()
df["gender"]          = le_gender.fit_transform(df["gender"])
df["smoking_history"] = le_smoking.fit_transform(df["smoking_history"])

# Features and target
FEATURE_COLS = ["gender", "age", "hypertension", "heart_disease",
                "smoking_history", "bmi", "HbA1c_level", "blood_glucose_level"]
TARGET_COL   = "diabetes"

X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32)

# Global train/test split (test set is held out — not seen by any client)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Total samples : {len(df):,}")
print(f"Train samples : {len(X_train):,}  |  Test samples : {len(X_test):,}")
print(f"Positive rate : {y.mean():.3f}  (class imbalance check)")
print(f"Features      : {FEATURE_COLS}")

Total samples : 100,000
Train samples : 85,000  |  Test samples : 15,000
Positive rate : 0.085  (class imbalance check)
Features      : ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']


In [16]:
# ── Client Data Partitioning (IID) ────────────────────────────────────────────
NUM_CLIENTS = 5

def iid_partition(X, y, num_clients, seed=42):
    """Split training data equally and randomly across clients (IID)."""
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    splits  = np.array_split(indices, num_clients)
    return [(X[idx], y[idx]) for idx in splits]

client_datasets = iid_partition(X_train, y_train, NUM_CLIENTS)
for i, (cx, cy) in enumerate(client_datasets):
    print(f"Client {i}: {len(cx):,} samples  |  positive rate: {cy.mean():.3f}")

Client 0: 17,000 samples  |  positive rate: 0.082
Client 1: 17,000 samples  |  positive rate: 0.086
Client 2: 17,000 samples  |  positive rate: 0.084
Client 3: 17,000 samples  |  positive rate: 0.089
Client 4: 17,000 samples  |  positive rate: 0.084


In [17]:
# ── Neural Network Model ───────────────────────────────────────────────────────
INPUT_DIM = len(FEATURE_COLS)

class DiabetesMLP(nn.Module):
    """Simple MLP for binary diabetes classification."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def model_to_numpy(model: nn.Module) -> Dict[str, np.ndarray]:
    return {k: v.cpu().detach().numpy() for k, v in model.state_dict().items()}

def numpy_to_model(model: nn.Module, weight_dict: Dict[str, np.ndarray]) -> nn.Module:
    sd = OrderedDict({k: torch.tensor(v) for k, v in weight_dict.items()})
    model.load_state_dict(sd)
    return model

# Verify model shape
_m = DiabetesMLP(INPUT_DIM)
print(_m)
print(f"Parameters: {sum(p.numel() for p in _m.parameters()):,}")

DiabetesMLP(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
    (6): Sigmoid()
  )
)
Parameters: 2,689


In [18]:
# ── Client Local Training ──────────────────────────────────────────────────────
def client_train(
    global_weights: Dict[str, np.ndarray],
    X_client: np.ndarray,
    y_client: np.ndarray,
    local_epochs: int = 3,
    batch_size: int = 64,
    lr: float = 1e-3,
) -> Tuple[Dict[str, np.ndarray], int]:
    """
    Train a local model starting from global_weights.

    Returns:
        (updated_weights, num_samples)
    """
    model = DiabetesMLP(INPUT_DIM)
    model = numpy_to_model(model, global_weights)
    model.train()

    X_t = torch.tensor(X_client, dtype=torch.float32)
    y_t = torch.tensor(y_client, dtype=torch.float32)
    dataset = TensorDataset(X_t, y_t)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for _ in range(local_epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss  = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    return model_to_numpy(model), len(X_client)

In [19]:
# ── Federated Learning Training Loop ──────────────────────────────────────────
FL_ROUNDS    = 10
LOCAL_EPOCHS = 3

def evaluate(weights: Dict[str, np.ndarray], X_eval, y_eval, threshold=0.5):
    """Evaluate global model on a held-out set."""
    model = DiabetesMLP(INPUT_DIM)
    model = numpy_to_model(model, weights)
    model.eval()
    with torch.no_grad():
        X_t   = torch.tensor(X_eval, dtype=torch.float32)
        probs = model(X_t).numpy()
    preds = (probs >= threshold).astype(int)
    return accuracy_score(y_eval, preds)

# Initialise global model
global_model = DiabetesMLP(INPUT_DIM)
init_weights = model_to_numpy(global_model)

server = FederatedServer(init_weights)
client_ids = [f"client_{i}" for i in range(NUM_CLIENTS)]

# Seed each client with the initial global model
current_global = {cid: server.get_global_model() for cid in client_ids}

round_accuracies = []

for fl_round in range(1, FL_ROUNDS + 1):
    print(f"\n── Round {fl_round} ──────────────────────────────────")

    # 1. Each client trains locally using its copy of the global model
    client_updates = []
    for i, cid in enumerate(client_ids):
        X_c, y_c = client_datasets[i]
        updated_w, n_samples = client_train(
            current_global[cid], X_c, y_c, local_epochs=LOCAL_EPOCHS
        )
        client_updates.append((updated_w, n_samples))
        print(f"  [{cid}] Local training done  ({n_samples:,} samples)")

    # 2. Server aggregates updates, then redistributes the new global model
    current_global = server.update_and_redistribute(client_updates, client_ids)

    # 3. Evaluate updated global model on held-out test set
    acc = evaluate(server.global_weights, X_test, y_test)
    round_accuracies.append(acc)
    print(f"  [Eval ] Test Accuracy: {acc:.4f}")

print("\nFederated Training Complete.")
server.summary()



── Round 1 ──────────────────────────────────
  [client_0] Local training done  (17,000 samples)
  [client_1] Local training done  (17,000 samples)
  [client_2] Local training done  (17,000 samples)
  [client_3] Local training done  (17,000 samples)
  [client_4] Local training done  (17,000 samples)
  [Server] Global model updated (round 1, 5 clients, 85,000 samples)
  [Server] Updated weights redistributed → ['client_0', 'client_1', 'client_2', 'client_3', 'client_4']
  [Eval ] Test Accuracy: 0.9599

── Round 2 ──────────────────────────────────
  [client_0] Local training done  (17,000 samples)
  [client_1] Local training done  (17,000 samples)
  [client_2] Local training done  (17,000 samples)
  [client_3] Local training done  (17,000 samples)
  [client_4] Local training done  (17,000 samples)
  [Server] Global model updated (round 2, 5 clients, 85,000 samples)
  [Server] Updated weights redistributed → ['client_0', 'client_1', 'client_2', 'client_3', 'client_4']
  [Eval ] Test Acc

In [20]:
# ── Final Evaluation & Report ──────────────────────────────────────────────────
final_model = DiabetesMLP(INPUT_DIM)
numpy_to_model(final_model, server.get_global_model())
final_model.eval()

with torch.no_grad():
    X_t   = torch.tensor(X_test, dtype=torch.float32)
    probs = final_model(X_t).numpy()

preds = (probs >= 0.5).astype(int)
print("Final Global Model — Test Set Report")
print("=" * 45)
print(classification_report(y_test, preds, target_names=["No Diabetes", "Diabetes"]))

# Accuracy over rounds
print("Accuracy per FL round:")
for r, acc in enumerate(round_accuracies, 1):
    bar = "█" * int(acc * 40)
    print(f"  Round {r:2d}: {acc:.4f}  {bar}")

Final Global Model — Test Set Report
              precision    recall  f1-score   support

 No Diabetes       0.97      1.00      0.98     13725
    Diabetes       0.99      0.67      0.80      1275

    accuracy                           0.97     15000
   macro avg       0.98      0.83      0.89     15000
weighted avg       0.97      0.97      0.97     15000

Accuracy per FL round:
  Round  1: 0.9599  ██████████████████████████████████████
  Round  2: 0.9624  ██████████████████████████████████████
  Round  3: 0.9655  ██████████████████████████████████████
  Round  4: 0.9682  ██████████████████████████████████████
  Round  5: 0.9699  ██████████████████████████████████████
  Round  6: 0.9702  ██████████████████████████████████████
  Round  7: 0.9709  ██████████████████████████████████████
  Round  8: 0.9715  ██████████████████████████████████████
  Round  9: 0.9715  ██████████████████████████████████████
  Round 10: 0.9714  ██████████████████████████████████████
